# ARS Notebook 02 — GSM8K In-Distribution Benchmark

Run one shot condition per Kaggle session across the four paired arms: baseline, fixed-route steering, validation-selected static bank, and router. The default is **0-shot on all 1,319 official GSM8K test questions**. To run 2-shot or 4-shot on the full test split, change only `SHOT_COUNT` in the configuration cell. Each run uses the same seed and question subset across arms, prints the paired accuracy/performance/quality summary, and saves a result ZIP whose name records the dataset, question count, and completed shot conditions.


## Input contract

For the first run, attach exactly one complete `ars_training_bundle.zip` (Stage B) or its extracted Kaggle Model directory with the root `manifest.json`. **Do not upload the Stage A ZIP**: Stage B already contains the fixed adapter, route bank, router, calibrations, and selected static action. No rollout files are needed.

For later shot runs, attach the same Stage B bundle and the latest cumulative GSM8K result ZIP to preserve earlier conditions in the summary. The notebook validates the training manifest and artifact hashes, and it refuses incomplete four-arm records.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
from scripts.research_notebook_utils import (
    extract_zip_once, locate_named_file, merge_result_bundle, pack_files,
    resolve_artifacts, run_live, sha256_file, atomic_write_json,
)

# Keep the training and evaluation revisions explicit. A stale script snapshot
# inside the training bundle must not replace the current benchmark runner.
EVALUATION_CODE_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
training_zips = sorted(INPUT_ROOT.rglob("ars_training_bundle.zip"))
if len(training_zips) > 1:
    raise ValueError(f"Attach one training bundle only; found: {[str(p) for p in training_zips]}")
TRAINING_ZIP = training_zips[0] if training_zips else None
TRAINING_INPUT = None
TRAINING_MANIFEST = {}
if TRAINING_ZIP:
    ARTIFACT_ROOT = extract_zip_once(TRAINING_ZIP, WORK_ROOT / "ars_training_artifacts")
    TRAINING_INPUT = TRAINING_ZIP
    manifest_path = ARTIFACT_ROOT / "manifest.json"
    if manifest_path.is_file():
        TRAINING_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
else:
    manifests = []
    for candidate in INPUT_ROOT.rglob("manifest.json"):
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
        except (OSError, ValueError, TypeError):
            continue
        if data.get("format") == "ars_training_bundle_v1":
            manifests.append((candidate, data))
    if len(manifests) > 1:
        raise ValueError(f"Attach one extracted ARS training bundle only; found: {[str(p) for p, _ in manifests]}")
    if manifests:
        manifest_path, TRAINING_MANIFEST = manifests[0]
        ARTIFACT_ROOT = manifest_path.parent
        TRAINING_INPUT = ARTIFACT_ROOT
    else:
        ARTIFACT_ROOT = INPUT_ROOT
        manifest_path = None

REQUIRED_ARTIFACTS = [
    "fixed_route_adapter.pt", "session3_steering_bank.pt", "session3_router.pt",
    "fixed_route_p99_norms.json", "candidate_norms_p99.json",
    "session3_static_bank_action.json",
]
try:
    ARTIFACTS = resolve_artifacts(ARTIFACT_ROOT, REQUIRED_ARTIFACTS)
    INPUTS_READY = bool(TRAINING_INPUT)
except Exception as exc:
    ARTIFACTS, INPUTS_READY = {}, False
    print("Training bundle is missing or invalid:", exc, flush=True)
if not TRAINING_INPUT:
    INPUTS_READY = False
    print("Attach ars_training_bundle.zip or one extracted input containing its root manifest.json.", flush=True)
elif (
    not TRAINING_MANIFEST.get("training_ready", False)
    or TRAINING_MANIFEST.get("architecture_id") != "ars.fixed_route_low_rank_residual"
    or TRAINING_MANIFEST.get("architecture_version") != "1.0.0"
):
    INPUTS_READY = False
    print("Training bundle is not ready or has an unsupported fixed-adapter architecture.", TRAINING_MANIFEST.get("required_phase_status"), flush=True)
TRAINING_CODE_COMMIT = TRAINING_MANIFEST.get("code_commit")
TRAINING_INPUT_SHA256 = (
    sha256_file(TRAINING_ZIP) if TRAINING_ZIP else
    sha256_file(manifest_path) if TRAINING_INPUT and manifest_path and manifest_path.is_file() else None
)
EXPECTED_ARTIFACT_SHA256 = (
    {
        "fixed_adapter_checkpoint": sha256_file(ARTIFACTS["fixed_route_adapter.pt"]),
        "fixed_adapter_norms": sha256_file(ARTIFACTS["fixed_route_p99_norms.json"]),
        "bank_checkpoint": sha256_file(ARTIFACTS["session3_steering_bank.pt"]),
        "router_checkpoint": sha256_file(ARTIFACTS["session3_router.pt"]),
        "candidate_norms": sha256_file(ARTIFACTS["candidate_norms_p99.json"]),
        "static_bank_action": sha256_file(ARTIFACTS["session3_static_bank_action.json"]),
    }
    if INPUTS_READY else {}
)
ARTIFACT_FINGERPRINT = hashlib.sha256(
    json.dumps(EXPECTED_ARTIFACT_SHA256, sort_keys=True).encode("utf-8")
).hexdigest()[:12] if INPUTS_READY else "missing"
print({
    "repository": str(REPO_DIR),
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "training_code_commit": TRAINING_CODE_COMMIT,
    "training_input": str(TRAINING_INPUT) if TRAINING_INPUT else None,
    "input_mode": "zip" if TRAINING_ZIP else ("extracted_directory" if TRAINING_INPUT else "missing"),
    "artifacts_ready": INPUTS_READY,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "artifacts": {key: str(value) for key, value in ARTIFACTS.items()},
    "gpus": subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines(),
}, flush=True)


## Choose one shot condition per session

The configuration cell defaults to `SHOT_COUNT = 0` and `BENCH_N = 1319` (the full official test split). For a separate full-data run at 2-shot or 4-shot, change `SHOT_COUNT` to `2` or `4` and rerun the notebook. Based on the previous 700-question four-arm benchmark on one Tesla T4, budget about **7–8.5 hours for 0-shot**, **5–6 hours for 2-shot**, and **5–7 hours for 4-shot** (the 4-shot estimate has not been measured directly). Run each shot condition in a separate Kaggle session and download the resulting ZIP before starting the next run.

The output ZIP name lists all complete conditions it contains, for example `ars_gsm8k_n1319_shots_0_results.zip` and then `ars_gsm8k_n1319_shots_0-2_results.zip`. Upload the latest cumulative ZIP alongside the Stage B bundle for the next shot condition.


In [ ]:
SEED = 42
FULL_GSM8K_TEST_N = 1319
SHOT_COUNT = 0  # Change to 2 or 4 for another full-data run.
BENCH_N = FULL_GSM8K_TEST_N  # Keep 1319 to evaluate the complete official test split.
if not isinstance(SHOT_COUNT, int) or not 0 <= SHOT_COUNT <= 8:
    raise ValueError("SHOT_COUNT must be an integer from 0 through 8")
if not isinstance(BENCH_N, int) or not 1 <= BENCH_N <= FULL_GSM8K_TEST_N:
    raise ValueError(f"BENCH_N must be between 1 and {FULL_GSM8K_TEST_N}")
if not INPUTS_READY:
    print("Evaluation is blocked until a complete Stage B training bundle is attached.", flush=True)
RESULTS = WORK_ROOT / "ars_gsm8k_results"
RESULTS.mkdir(parents=True, exist_ok=True)
previous_bundles = sorted({p for p in INPUT_ROOT.rglob("ars_gsm8k*_results.zip") if p.is_file()})
if len(previous_bundles) > 1:
    raise ValueError(f"Attach only the latest cumulative GSM8K result ZIP; found: {[str(p) for p in previous_bundles]}")
if previous_bundles:
    merge_result_bundle(previous_bundles[0], RESULTS)

print({
    "dataset": "GSM8K official test",
    "requested_questions": BENCH_N,
    "shot_count": SHOT_COUNT,
    "seed": SEED,
    "arms": ["baseline", "steer_only", "static_bank", "router"],
    "training_bundle_ready": INPUTS_READY,
    "previous_result_bundle": str(previous_bundles[0]) if previous_bundles else None,
}, flush=True)

def completed(shot, n):
    if not INPUTS_READY:
        return None
    for path in RESULTS.rglob(f"session3_gsm8k_{n}_seed{SEED}.json"):
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
            if (
                record.get("suite") == "gsm8k"
                and record.get("test_seed") == SEED
                and len(record.get("test_source_indices", [])) == n
                and set(record.get("shots", {})) == {str(shot)}
                and record.get("four_arm_complete")
                and record.get("artifact_sha256") == EXPECTED_ARTIFACT_SHA256
                and record.get("provenance", {}).get("evaluation_code_commit") == EVALUATION_CODE_COMMIT
            ):
                return path
        except (OSError, ValueError, TypeError):
            continue
    return None

def annotate_record(path):
    if not path.is_file():
        return
    record = json.loads(path.read_text(encoding="utf-8"))
    record["provenance"] = {
        "training_code_commit": TRAINING_CODE_COMMIT,
        "evaluation_code_commit": EVALUATION_CODE_COMMIT,
        "training_input_sha256": TRAINING_INPUT_SHA256,
        "artifact_fingerprint": ARTIFACT_FINGERPRINT,
        "primary_shots": [0, 2],
    }
    record["artifact_sha256"] = EXPECTED_ARTIFACT_SHA256
    atomic_write_json(path, record)

def run_condition(shot, n):
    old = completed(shot, n)
    if old:
        print("COMPLETE for these exact artifacts and code; skip:", old, flush=True)
        return
    if not INPUTS_READY:
        print("SKIP: missing or invalid Stage B training bundle", flush=True)
        return
    out = RESULTS / ARTIFACT_FINGERPRINT / EVALUATION_CODE_COMMIT[:12] / f"{shot}shot_n{n}"
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ARTIFACTS["session3_static_bank_action.json"], out / "session3_static_bank_action.json")
    cmd = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "benchmark",
           "--output_dir", str(out), "--candidate_layers", "8", "11", "12", "13", "15", "17", "18", "19", "20", "21", "22", "23", "24", "26",
           "--norm_calibration", str(ARTIFACTS["candidate_norms_p99.json"]),
           "--fixed_adapter_norm_calibration", str(ARTIFACTS["fixed_route_p99_norms.json"]),
           "--fixed_adapter_checkpoint", str(ARTIFACTS["fixed_route_adapter.pt"]),
           "--bank_checkpoint", str(ARTIFACTS["session3_steering_bank.pt"]),
           "--router_checkpoint", str(ARTIFACTS["session3_router.pt"]),
           "--bench_n", str(n), "--bench_seed", str(SEED), "--shots", str(shot),
           "--require_all_arms", "--diversify_prompts"]
    code = run_live(cmd, cwd=REPO_DIR, log_path=out / "benchmark.log")
    record_path = out / f"session3_gsm8k_{n}_seed{SEED}.json"
    if code == 0 and record_path.is_file():
        annotate_record(record_path)
    print({"shot": shot, "n": n, "exit_code": code, "record": str(record_path), "complete": bool(completed(shot, n))}, flush=True)

run_condition(SHOT_COUNT, BENCH_N)

from IPython.display import display
import pandas as pd
from scripts.research_notebook_analysis import build_benchmark_summary

metrics_summary = build_benchmark_summary(
    [RESULTS], RESULTS / "analysis",
    artifact_sha256=EXPECTED_ARTIFACT_SHA256,
    artifact_fingerprint=ARTIFACT_FINGERPRINT,
    evaluation_code_commit=EVALUATION_CODE_COMMIT,
)
metrics = metrics_summary["arm_metrics"]
if metrics:
    metrics_table = pd.DataFrame(metrics)
    performance_columns = [
        "suite", "shot", "arm", "n", "accuracy_pct",
        "paired_delta_pp_vs_baseline", "paired_ci95_pp_vs_baseline", "mcnemar_p_value_vs_baseline",
        "n_timed", "timing_coverage_rate_pct", "mean_generation_latency_seconds", "median_generation_latency_seconds",
        "p90_generation_latency_seconds", "total_generation_seconds",
        "questions_per_second", "output_tokens_per_second", "mean_prompt_tokens", "mean_new_tokens", "timed_output_tokens",
        "router_unique_routes", "router_most_used_route_share", "router_k_counts", "device_name",
    ]
    quality_columns = [
        "suite", "shot", "arm", "n", "strict_accuracy_pct", "marker_accuracy_pct",
        "lenient_accuracy_pct", "extraction_rate_pct", "max_new_tokens_rate_pct", "stop_reason_counts",
        "equation_coverage_rate_pct", "mean_eaa_when_equations_present",
        "clean_arithmetic_rate_pct", "guard_trip_rate_pct", "nonfinite_count",
    ]
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 2200)
    print("Paired accuracy and performance (latency in seconds; rates are per model-generation call):")
    display(metrics_table[[name for name in performance_columns if name in metrics_table]].round(3))
    print("Answer quality, output format, arithmetic diagnostics, and safety checks:")
    display(metrics_table[[name for name in quality_columns if name in metrics_table]].round(3))
    paired_table = pd.DataFrame(metrics_summary["paired_comparisons"])
    if not paired_table.empty:
        print("Paired arm-to-arm comparisons:")
        display(paired_table[[name for name in ["suite", "shot", "reference_arm", "arm", "n", "wins", "losses", "ties", "delta_pp", "ci95_pp", "mcnemar_p_value"] if name in paired_table]].round(4))
    route_table = pd.DataFrame(metrics_summary["route_usage"])
    if not route_table.empty:
        print("Router route, k, layer, and gain usage:")
        display(route_table)
else:
    print("No complete four-arm records for the current artifacts and evaluation-code revision yet.")

# Build an explicit condition list so the archive name and manifest state exactly
# which full-data shot runs are complete and which condition was requested now.
complete_conditions = set()
for path in RESULTS.rglob("session3_gsm8k_*.json"):
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
        count = len(record.get("test_source_indices", []))
        if (
            record.get("suite") == "gsm8k"
            and record.get("four_arm_complete")
            and record.get("artifact_sha256") == EXPECTED_ARTIFACT_SHA256
            and record.get("provenance", {}).get("evaluation_code_commit") == EVALUATION_CODE_COMMIT
        ):
            complete_conditions.update((count, int(shot)) for shot in record.get("shots", {}))
    except (OSError, ValueError, TypeError):
        continue
current_complete = completed(SHOT_COUNT, BENCH_N) is not None
bundle_manifest = {
    "format": "ars_gsm8k_result_bundle_v1",
    "suite": "gsm8k",
    "dataset": "GSM8K official test split",
    "requested_condition": {"n": BENCH_N, "shots": [SHOT_COUNT], "seed": SEED},
    "arms": ["baseline", "steer_only", "static_bank", "router"],
    "bundle_status": "complete" if current_complete else "partial",
    "complete_conditions_included": [
        {"n": n, "shots": [shot]} for n, shot in sorted(complete_conditions)
    ],
    "training_code_commit": TRAINING_CODE_COMMIT,
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "training_input_sha256": TRAINING_INPUT_SHA256,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "summary_record_count": metrics_summary["record_count"],
    "summary_files": [
        "analysis/benchmark_metrics_summary.json",
        "analysis/benchmark_metrics_summary.csv",
        "analysis/benchmark_paired_comparisons.csv",
        "analysis/benchmark_route_usage.csv",
    ],
}
manifest_path = RESULTS / "bundle_manifests" / f"gsm8k_n{BENCH_N}_{SHOT_COUNT}shot_seed{SEED}.json"
atomic_write_json(manifest_path, bundle_manifest)

files = {p: str(p.relative_to(RESULTS)) for p in RESULTS.rglob("*") if p.is_file() and p.name != ".source_sha256"}
for name in ("run_session3_router.py", "run_session3_mcq_benchmark.py", "research_notebook_utils.py", "research_notebook_analysis.py"):
    path = REPO_DIR / "scripts" / name
    archive_name = f"code/scripts/{name}"
    if path.is_file() and archive_name not in files.values():
        files[path] = archive_name
for relative in ("src/evaluation/benchmark.py", "src/evaluation/answers.py", "src/evaluation/ood.py"):
    path = REPO_DIR / relative
    archive_name = f"code/{relative}"
    if path.is_file() and archive_name not in files.values():
        files[path] = archive_name

if complete_conditions:
    counts = sorted({n for n, _ in complete_conditions})
    if len(counts) == 1:
        shot_label = "-".join(str(shot) for _, shot in sorted(complete_conditions))
        bundle_name = f"ars_gsm8k_n{counts[0]}_shots_{shot_label}_results.zip"
    else:
        condition_label = "__".join(f"n{n}_{shot}shot" for n, shot in sorted(complete_conditions))
        bundle_name = f"ars_gsm8k_{condition_label}_results.zip"
else:
    bundle_name = f"ars_gsm8k_n{BENCH_N}_{SHOT_COUNT}shot_partial_results.zip"
if not current_complete and complete_conditions:
    bundle_name = bundle_name.removesuffix("_results.zip") + f"_plus_n{BENCH_N}_{SHOT_COUNT}shot_partial_results.zip"
zip_path = pack_files(files, WORK_ROOT / bundle_name) if files else None
print({
    "requested_condition_complete": current_complete,
    "complete_conditions": [{"n": n, "shot": shot} for n, shot in sorted(complete_conditions)],
    "summary": str(RESULTS / "analysis" / "benchmark_metrics_summary.json"),
    "result_bundle": str(zip_path) if zip_path else None,
    "bundle_manifest": str(manifest_path),
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
}, flush=True)


## Output

Each condition saves its aggregate JSON, paired generation JSONL, per-arm generations, report, and execution log. The cumulative ZIP has a descriptive GSM8K/test-count/shot filename; attach the latest ZIP to the next run to retain prior completed conditions. Matching complete conditions are skipped rather than generated again.

The final cell displays accuracy and performance, answer-quality diagnostics, all paired arm comparisons, and router-use tables. The archive includes `analysis/benchmark_metrics_summary.csv`, `benchmark_paired_comparisons.csv`, `benchmark_route_usage.csv`, `benchmark_metrics_summary.json`, and a per-run bundle manifest. Every arm record retains per-question timing and token counts.
